# Bài toán 3 — Thảo luận, demo và báo cáo

## 9. Giải thích và thảo luận

Mô hình được chọn nhân mức Revenue hoặc COGS trung bình ngày của năm đầy đủ gần nhất với hệ số mùa vụ của cặp tháng–ngày. Vì mức gốc cố định, ngày có cùng vị trí trên lịch nhận cùng dự báo giữa các năm dự báo. Phương pháp dễ giải thích và bỏ qua biến động tức thời hoặc thay đổi mức trong kỳ.

Ngày 02/03/2022 bị dự báo thiếu nhiều nhất ở Revenue, khoảng 4,105 triệu. Ngày 30/03/2022 bị dự báo thiếu nhiều nhất ở COGS, khoảng 3,712 triệu. Những chênh lệch này cho thấy hạn chế khi dự báo các ngày tăng mạnh. Ba cột của `sales.csv` chỉ cho phép xác định độ lệch và thời điểm, chưa xác định được nguyên nhân nghiệp vụ. Trong validation, phương pháp giữ mức gốc cố định tốt hơn ở hai lượt đầu, còn baseline có tăng trưởng tốt hơn ở lượt 3.

## 10. Demo dự báo trong notebook

Đổi hai mốc ngày để xem dự báo Revenue và COGS trong một khoảng của tập test. Có thể dùng ô này khi trình bày. Ứng dụng triển khai riêng là bước tùy chọn trong quy trình.

Dữ liệu demo được đọc từ kết quả của notebook 02.

In [1]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

sales_path = Path("data/raw/sales.csv")
start = Path(os.environ.get("RETAIL_ANALYTICS_ROOT", Path.cwd())).resolve()
project_root = next(
    (folder for folder in (start, *start.parents) if (folder / sales_path).is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Không tìm thấy data/raw/sales.csv. Hãy đặt notebook trong dự án "
        "hoặc trỏ RETAIL_ANALYTICS_ROOT về thư mục gốc."
    )

prediction_path = project_root / "data/model/du_bao/forecast_predictions.csv"
if not prediction_path.is_file():
    raise FileNotFoundError("Chạy 02_mo_hinh.ipynb để tạo forecast_predictions.csv.")

predictions = pd.read_csv(
    prediction_path, parse_dates=["Date", "TrainEnd"], float_precision="round_trip",
)
targets = ["Revenue", "COGS"]

contest_name = "Baseline cuộc thi"
fixed_name = "Mùa vụ, mức gốc cố định"
selected_label = "Phương pháp được chọn"


def prediction_frame(rows):
    return rows.set_index("Date")[targets].sort_index()


def test_forecast_for(method):
    return prediction_frame(predictions.loc[
        predictions["Split"].eq("test") & predictions["Method"].eq(method)
    ])

selected_test_forecast = test_forecast_for(selected_label)

demo_start = "2021-07-02"
demo_end = "2021-07-08"
display(selected_test_forecast.loc[demo_start:demo_end].rename(
    columns={"Revenue": "Revenue dự báo", "COGS": "COGS dự báo"},
))

,Revenue dự báo,COGS dự báo
Date,,
2021-07-02,4609905.25,4365213.06
2021-07-03,4097775.31,3909822.32
2021-07-04,1283312.22,1152652.84
2021-07-05,1551951.80,1431980.12
2021-07-06,1667077.43,1576759.59
2021-07-07,2020799.19,1911000.38
2021-07-08,1941426.53,1826584.36


## 11. Nội dung đưa vào báo cáo

Nhóm sử dụng 3.833 quan sát theo ngày để khảo sát biến động, xây dựng và so sánh các phương pháp dự báo Revenue và COGS. Ba lượt validation, mỗi lượt 548 ngày, được dùng để chọn mô hình theo MAE. Hệ số được ước lượng riêng từ tập train của từng lượt.

Trên 548 ngày test từ 02/07/2021 đến 31/12/2022, mô hình mùa vụ với mức gốc cố định đạt MAE **0,596** cho Revenue và **0,542** cho COGS, giảm lần lượt **13,7%** và **20,2%** so với baseline cuộc thi trên cùng kỳ. RMSE lần lượt là **0,821** và **0,752** triệu đơn vị tiền tệ/ngày. R² lần lượt là **0,721** và **0,697**. Kết luận về cải thiện áp dụng cho kỳ kiểm tra này. Bảng đánh giá ở mục 7 và biểu đồ, sai số theo ngày ở mục 8 là minh chứng cho báo cáo.

## 12. Thuyết trình và vấn đáp

**Mạch trình bày.** Nêu hai chỉ tiêu và mốc dự báo, chỉ cách tách validation/test, đọc bảng so sánh, rồi mở biểu đồ và chọn một ngày sai nhiều để giải thích hạn chế.

| Câu hỏi có thể gặp | Cách giải thích |
|---|---|
| Vì sao chia theo thời gian? | Khi dự báo tại một mốc, chỉ lịch sử trước mốc mới có thể dùng để tính hệ số. Tách theo thời gian tái hiện điều kiện đó ở từng lượt. |
| Vì sao chọn phương pháp mùa vụ với mức gốc cố định? | Nó có MAE trung bình thấp nhất ở cả hai chỉ tiêu trên ba lượt validation. Kết quả từng lượt cho thấy thứ hạng thay đổi theo giai đoạn. |
| Vì sao có cả MAE, RMSE và R²? | MAE cho biết sai số điển hình tính theo đơn vị gốc. RMSE nhấn mạnh lỗi lớn. R² so sánh với mức tham chiếu trung bình thực tế. |
| Mô hình có hạn chế gì? | Nó lặp lại quy luật tháng–ngày và giữ mức gốc cố định, nên có thể dự báo thiếu khi số thực tế tăng mạnh. |